<a href="https://colab.research.google.com/github/ISHANTIITM/CompressWebsite/blob/main/Copy_of_notebook72c503eb0a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup: imports, seeds, switches, load train/test CSVs, compute baseline RMSE

In [ ]:
import os, re, pickle, warnings, random, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa, torch

from tqdm.auto import tqdm
from scipy.stats import pearsonr, loguniform, randint

from sklearn.model_selection import StratifiedKFold, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression, HuberRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, make_scorer, log_loss

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR = 16000

# ---------------- switches ----------------
VERBATIM_ASR = False   # True = re-run Whisper with a disfluency prompt (~78 min, new cache file)
FAST = False           # True = smaller grids for quick experiments
# ------------------------------------------

ROOT = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

train = pd.read_csv(f"{ROOT}/train.csv")
test = pd.read_csv(f"{ROOT}/test.csv")
sample_sub = pd.read_csv(f"{ROOT}/sample_submission.csv")

y = train["label"].values.astype(float)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

base_rmse = rmse(y, np.full(len(y), y.mean()))

print("Train:", train.shape)
print("Test :", test.shape)
print("Device:", DEVICE)
print("Baseline RMSE:", round(base_rmse, 4))

EDA: plot label distribution and score counts

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train["label"], bins=20, ax=ax[0])
ax[0].set_title("Label distribution")
train["label"].value_counts().sort_index().plot.bar(ax=ax[1])
ax[1].set_title("Counts per score")
plt.tight_layout()
plt.show()

print(train["label"].describe())
print("Baseline RMSE:", round(base_rmse, 4))

Transcribe all audio with Whisper-medium (cached) and attach text to train/test

In [ ]:
from transformers import pipeline

ASR_MODEL = "openai/whisper-medium"
ASR_BATCH_SIZE = 8

CACHE = (
    f"{WORK}/transcripts_whisper_medium_verbatim.pkl"
    if VERBATIM_ASR
    else f"{WORK}/transcripts_whisper_medium.pkl"
)

if os.path.exists(CACHE):
    with open(CACHE, "rb") as f:
        trans = pickle.load(f)
else:
    trans = {}

todo_any = any(
    (split, f) not in trans
    for split, df in [("train", train), ("test", test)]
    for f in df.filename
)

if todo_any:
    asr = pipeline(
        "automatic-speech-recognition",
        model=ASR_MODEL,
        device=0 if DEVICE == "cuda" else -1,
        dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    )

    GEN = {"language": "en", "task": "transcribe"}
    if VERBATIM_ASR:
        prompt_ids = asr.tokenizer.get_prompt_ids(
            "Umm, let me think, like, hmm... I mean, yeah, so, uh, I was going to say.",
            return_tensors="pt"
        ).to(DEVICE)
        GEN["prompt_ids"] = prompt_ids

    def run_split(split, df, bs=ASR_BATCH_SIZE):
        todo = [f for f in df.filename if (split, f) not in trans]

        for i in tqdm(range(0, len(todo), bs), desc=f"Whisper {split}"):
            fns = todo[i:i + bs]

            audios = [
                librosa.load(f"{ROOT}/{split}/{f}", sr=SR, mono=True)[0]
                for f in fns
            ]

            silent = [np.abs(a).max() < 1e-4 for a in audios]

            batch = [
                {"raw": a, "sampling_rate": SR}
                for a, is_silent in zip(audios, silent)
                if not is_silent
            ]

            outs = iter(
                asr(batch, batch_size=bs, chunk_length_s=30, generate_kwargs=GEN)
                if batch else []
            )

            for f, is_silent in zip(fns, silent):
                trans[(split, f)] = "" if is_silent else next(outs)["text"].strip()

            with open(CACHE, "wb") as fp:
                pickle.dump(trans, fp)

    run_split("train", train)
    run_split("test", test)

train["text"] = [trans[("train", f)] for f in train.filename]
test["text"] = [trans[("test", f)] for f in test.filename]

print("Empty transcripts:", "train =", (train.text == "").sum(), "test =", (test.text == "").sum())
display(train[["filename", "label", "text"]].head(10))

Bucket scores into 5 classes and compute class centers for class-to-score conversion

In [ ]:
def make_cls(x):
    return (
        0 if x <= 1.5 else
        1 if x <= 2.5 else
        2 if x <= 3.5 else
        3 if x <= 4.5 else
        4
    )

train["cls"] = train["label"].apply(make_cls)
y_cls = train["cls"].values
classes = np.sort(np.unique(y_cls))

centers = np.array([train.loc[train["cls"] == c, "label"].mean() for c in classes])

def to_score(P):
    return np.clip(P @ centers, 0, 5)

print("Class counts:", train["cls"].value_counts().sort_index().to_dict())
print("Class centers:", np.round(centers, 3))

Build text features (word stats, CoLA grammar scores, GPT-2 perplexity) and plot their correlation with label

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2LMHeadModel,
    GPT2TokenizerFast
)

dur_tr = np.array([librosa.get_duration(path=f"{ROOT}/train/{f}") for f in train.filename])
dur_te = np.array([librosa.get_duration(path=f"{ROOT}/test/{f}") for f in test.filename])

cola_tok = AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
cola = AutoModelForSequenceClassification.from_pretrained(
    "textattack/roberta-base-CoLA"
).to(DEVICE).eval()

def split_sents(t):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.split()) >= 2]

@torch.no_grad()
def cola_feats(texts, bs=64):
    sents, owner = [], []

    for i, t in enumerate(texts):
        for s in split_sents(t):
            sents.append(s)
            owner.append(i)

    if not sents:
        return pd.DataFrame({
            "cola_mean": np.zeros(len(texts)),
            "cola_min": np.zeros(len(texts)),
            "cola_bad_frac": np.zeros(len(texts))
        })

    probs = []

    for i in range(0, len(sents), bs):
        b = cola_tok(
            sents[i:i+bs], padding=True, truncation=True,
            max_length=64, return_tensors="pt"
        ).to(DEVICE)

        probs.extend(torch.softmax(cola(**b).logits, -1)[:, 1].cpu().numpy().tolist())

    tmp = pd.DataFrame({"i": owner, "p": probs})
    grp = tmp.groupby("i")["p"]

    out = pd.DataFrame({
        "cola_mean": grp.mean(),
        "cola_min": grp.min(),
        "cola_bad_frac": grp.apply(lambda s: (s < 0.5).mean())
    })

    return out.reindex(range(len(texts)))

gpt_tok = GPT2TokenizerFast.from_pretrained("gpt2")
gpt = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE).eval()

@torch.no_grad()
def log_ppl(t):
    if len(t.split()) < 3:
        return np.nan

    ids = gpt_tok(t, return_tensors="pt", truncation=True, max_length=512).input_ids.to(DEVICE)
    return float(gpt(ids, labels=ids).loss.item())

FILL = {"um", "uh", "er", "ah", "hmm", "erm", "uhm", "mm"}

def stat_feats(t, dur):
    w = re.findall(r"[a-zA-Z']+", t.lower())
    n = len(w)
    sents = [s for s in re.split(r"[.!?]+", t) if s.strip()]

    return dict(
        n_words=n,
        wpm=n / max(dur, 1) * 60,
        ttr=len(set(w)) / n if n else 0,
        avg_wlen=np.mean([len(x) for x in w]) if n else 0,
        n_sents=len(sents),
        avg_sent_len=n / len(sents) if sents else 0,
        filler_rate=sum(x in FILL for x in w) / max(n, 1),
        repeat_rate=sum(a == b for a, b in zip(w, w[1:])) / max(n, 1),
        comma_rate=t.count(",") / max(n, 1),
        log_ppl=log_ppl(t)
    )

def build_text_feats(df, dur):
    S = pd.DataFrame([
        stat_feats(t, d)
        for t, d in tqdm(zip(df.text, dur), total=len(df), desc="Text statistics")
    ])
    C = cola_feats(list(df.text)).reset_index(drop=True)
    return pd.concat([S, C], axis=1)

F_tr_text = build_text_feats(train, dur_tr)
F_te_text = build_text_feats(test, dur_te)

F_tr_text = F_tr_text.replace([np.inf, -np.inf], np.nan)
F_te_text = F_te_text.replace([np.inf, -np.inf], np.nan)

_med = F_tr_text.median()          # medians from raw train, computed once
F_tr_text = F_tr_text.fillna(_med)
F_te_text = F_te_text.fillna(_med)

corr = F_tr_text.assign(label=y).corr(numeric_only=True)["label"].drop("label")
corr.sort_values().plot.barh(figsize=(7, 6), title="Text/fluency feature correlation")
plt.tight_layout()
plt.show()

Extract RoBERTa mean-pooled embeddings of the transcripts (plus log word count)

In [ ]:
from transformers import AutoModel

tok = AutoTokenizer.from_pretrained("roberta-base")
enc = AutoModel.from_pretrained("roberta-base").to(DEVICE).eval()

@torch.no_grad()
def embed(texts, bs=16):
    out = []

    for i in range(0, len(texts), bs):
        batch_text = [t if isinstance(t, str) and t.strip() else "." for t in texts[i:i+bs]]

        b = tok(
            batch_text, padding=True, truncation=True,
            max_length=256, return_tensors="pt"
        ).to(DEVICE)

        h = enc(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1)

        pooled = (h * m).sum(1) / m.sum(1)
        out.append(pooled.cpu().numpy())

    return np.vstack(out)

E_tr_base = embed(list(train.text))
E_te_base = embed(list(test.text))

lw_tr = np.log1p(train.text.str.split().str.len().values)
lw_te = np.log1p(test.text.str.split().str.len().values)

E_tr = np.hstack([E_tr_base, lw_tr[:, None]])
E_te = np.hstack([E_te_base, lw_te[:, None]])

Xw_tr = pd.DataFrame({"text": train.text, "n_words": lw_tr})
Xw_te = pd.DataFrame({"text": test.text, "n_words": lw_te})

print("RoBERTa feature shape:", E_tr.shape)

Extract handcrafted acoustic features (pauses, MFCC, pitch/spectral stats) with librosa, cached

In [ ]:
def safe_stat(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return [0.0, 0.0, 0.0, 0.0, 0.0]

    return [
        float(np.mean(x)),
        float(np.std(x)),
        float(np.median(x)),
        float(np.percentile(x, 25)),
        float(np.percentile(x, 75))
    ]

def acoustic_one(path):
    y_audio, _ = librosa.load(path, sr=SR, mono=True)

    if len(y_audio) == 0:
        return {"duration": 0.0}

    duration = len(y_audio) / SR

    peak = np.max(np.abs(y_audio))
    y_norm = y_audio / max(peak, 1e-8)

    rms = librosa.feature.rms(y=y_norm, frame_length=1024, hop_length=256)[0]
    zcr = librosa.feature.zero_crossing_rate(y_norm, frame_length=1024, hop_length=256)[0]
    centroid = librosa.feature.spectral_centroid(y=y_norm, sr=SR, n_fft=1024, hop_length=256)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y_norm, sr=SR, n_fft=1024, hop_length=256)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y_norm, sr=SR, n_fft=1024, hop_length=256)[0]
    mfcc = librosa.feature.mfcc(y=y_norm, sr=SR, n_mfcc=13, n_fft=1024, hop_length=256)
    chroma = librosa.feature.chroma_stft(y=y_norm, sr=SR, n_fft=1024, hop_length=256)

    energy_threshold = max(np.percentile(rms, 15) * 1.5, 0.01)
    silent = rms < energy_threshold

    runs = []
    start = None

    for i, flag in enumerate(silent):
        if flag and start is None:
            start = i
        elif not flag and start is not None:
            runs.append((start, i - 1))
            start = None

    if start is not None:
        runs.append((start, len(silent) - 1))

    frame_hop_sec = 256 / SR

    pause_lengths = [
        (b - a + 1) * frame_hop_sec
        for a, b in runs
        if (b - a + 1) * frame_hop_sec >= 0.15
    ]

    pause_time = float(sum(pause_lengths))
    silence_ratio = pause_time / max(duration, 1e-6)

    feat = {
        "duration": duration,
        "silence_ratio": silence_ratio,
        "pause_count": float(len(pause_lengths)),
        "longest_pause": float(max(pause_lengths, default=0.0)),
        "mean_pause": float(np.mean(pause_lengths)) if pause_lengths else 0.0,
    }

    for prefix, arr in [("rms", rms), ("zcr", zcr), ("centroid", centroid),
                        ("bandwidth", bandwidth), ("rolloff", rolloff)]:
        vals = safe_stat(arr)
        for suffix, value in zip(["mean", "std", "median", "q25", "q75"], vals):
            feat[f"{prefix}_{suffix}"] = value

    for i, arr in enumerate(mfcc):
        vals = safe_stat(arr)
        for suffix, value in zip(["mean", "std", "median", "q25", "q75"], vals):
            feat[f"mfcc{i+1}_{suffix}"] = value

    for i, arr in enumerate(chroma):
        vals = safe_stat(arr)
        for suffix, value in zip(["mean", "std", "median", "q25", "q75"], vals):
            feat[f"chroma{i+1}_{suffix}"] = value

    return feat

def build_acoustic(split, df):
    return pd.DataFrame([
        acoustic_one(f"{ROOT}/{split}/{f}")
        for f in tqdm(df.filename, desc=f"Acoustic {split}")
    ])

AC_CACHE = f"{WORK}/acoustic_features.pkl"

if os.path.exists(AC_CACHE):
    with open(AC_CACHE, "rb") as f:
        acoustic_cache = pickle.load(f)
    ACOU_tr = acoustic_cache["train"]
    ACOU_te = acoustic_cache["test"]
else:
    ACOU_tr = build_acoustic("train", train)
    ACOU_te = build_acoustic("test", test)

    with open(AC_CACHE, "wb") as f:
        pickle.dump({"train": ACOU_tr, "test": ACOU_te}, f)

ACOU_tr = ACOU_tr.replace([np.inf, -np.inf], np.nan)
ACOU_te = ACOU_te.replace([np.inf, -np.inf], np.nan)

ac_medians = ACOU_tr.median(numeric_only=True)

ACOU_tr = ACOU_tr.fillna(ac_medians).astype(float)
ACOU_te = ACOU_te.fillna(ac_medians).astype(float)

print("Acoustic shape:", ACOU_tr.shape)

ac_corr = ACOU_tr.assign(label=y).corr(numeric_only=True)["label"].drop("label")
display(ac_corr.abs().sort_values(ascending=False).head(20))

Extract Wav2Vec2 hidden-state mean+std features for every layer, cached

In [ ]:
# CHANGE: wav2vec2 now pools mean + std per layer (new cache file name)
from transformers import Wav2Vec2Processor, Wav2Vec2Model

w_proc = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
w_mdl = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE).eval()

@torch.no_grad()
def audio_layers(split, df):
    out = []

    for f in tqdm(df.filename, desc=f"Wav2Vec2 {split}"):
        y_audio, _ = librosa.load(f"{ROOT}/{split}/{f}", sr=SR, mono=True)

        if len(y_audio) == 0:
            y_audio = np.zeros(SR, dtype=np.float32)

        x = w_proc(y_audio, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
        hs = w_mdl(x, output_hidden_states=True).hidden_states

        pooled = torch.stack([
            torch.cat([h[0].mean(0), h[0].std(0)])
            for h in hs
        ]).cpu().numpy()

        out.append(pooled)

    return np.stack(out)

AF = f"{WORK}/audio_layers_meanstd.npz"

if os.path.exists(AF):
    z = np.load(AF)
    AL_tr = z["tr"]
    AL_te = z["te"]
else:
    AL_tr = audio_layers("train", train)
    AL_te = audio_layers("test", test)
    np.savez(AF, tr=AL_tr, te=AL_te)

print("Wav2Vec2 layer tensor:", AL_tr.shape)

Extract Whisper-small encoder embeddings (chunked into 30 s windows), cached

In [ ]:
# NEW: Whisper encoder embeddings (mean+std of two layers, chunked into 30 s windows)
from transformers import WhisperFeatureExtractor, WhisperModel

WF = f"{WORK}/whisper_enc.npz"

if os.path.exists(WF):
    z = np.load(WF)
    WE_tr, WE_te = z["tr"], z["te"]
else:
    wfe = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
    wenc = WhisperModel.from_pretrained("openai/whisper-small").encoder.to(DEVICE).eval()

    @torch.no_grad()
    def whisper_enc(split, df, max_chunks=6):
        out = []
        for f in tqdm(df.filename, desc=f"WhisperEnc {split}"):
            a, _ = librosa.load(f"{ROOT}/{split}/{f}", sr=SR, mono=True)
            if len(a) == 0:
                a = np.zeros(SR, dtype=np.float32)

            chunk_feats = []
            for s in range(0, min(len(a), max_chunks * 30 * SR), 30 * SR):
                seg = a[s:s + 30 * SR]
                if len(seg) < SR // 2 and chunk_feats:
                    continue
                n_frames = max(int(np.ceil(len(seg) / SR * 50)), 2)   # 50 encoder frames / sec
                x = wfe(seg, sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE)
                hs = wenc(x, output_hidden_states=True).hidden_states
                parts = []
                for h in (hs[-4], hs[-1]):
                    v = h[0, :n_frames]
                    parts += [v.mean(0), v.std(0)]
                chunk_feats.append(torch.cat(parts).cpu().numpy())

            out.append(np.mean(chunk_feats, axis=0))
        return np.stack(out)

    WE_tr = whisper_enc("train", train)
    WE_te = whisper_enc("test", test)
    np.savez(WF, tr=WE_tr, te=WE_te)

print("Whisper encoder features:", WE_tr.shape)

Score each Wav2Vec2 layer with ridge CV, plot, and keep the 4 best layers

In [ ]:
kf_layer = KFold(5, shuffle=True, random_state=SEED)

layer_rmse = []

for L in range(AL_tr.shape[1]):
    X = AL_tr[:, L]
    oof_layer = np.zeros(len(y))

    for a, b in kf_layer.split(X):
        m = Pipeline([("sc", StandardScaler()), ("r", Ridge())])

        g = GridSearchCV(
            m,
            {"r__alpha": np.logspace(2, 6, 5)},      # wider range: features are 2x larger now
            cv=3,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1
        )

        g.fit(X[a], y[a])
        oof_layer[b] = g.predict(X[b])

    layer_rmse.append(rmse(y, np.clip(oof_layer, 0, 5)))

plt.figure(figsize=(8, 4))
plt.bar(range(len(layer_rmse)), layer_rmse)
plt.axhline(base_rmse, linestyle="--", label="mean baseline")
plt.xlabel("Wav2Vec2 layer")
plt.ylabel("5-fold RMSE")
plt.title("Audio layer informativeness")
plt.legend()
plt.show()

best_layers = sorted(np.argsort(layer_rmse)[:4])

print("Selected layers:", best_layers)

A_w2v_tr = AL_tr[:, best_layers].reshape(len(train), -1)
A_w2v_te = AL_te[:, best_layers].reshape(len(test), -1)

print("Selected Wav2Vec2 feature shape:", A_w2v_tr.shape)

Combine feature sets into JOINT (text+audio+acoustic) and QUALITY (text stats+acoustic) matrices

In [ ]:
JOINT_tr = np.hstack([E_tr, A_w2v_tr, ACOU_tr.values])
JOINT_te = np.hstack([E_te, A_w2v_te, ACOU_te.values])

QUALITY_tr = np.hstack([F_tr_text.values, ACOU_tr.values])
QUALITY_te = np.hstack([F_te_text.values, ACOU_te.values])

print("Text:", E_tr.shape)
print("Wav2Vec2:", A_w2v_tr.shape)
print("Whisper enc:", WE_tr.shape)
print("Acoustic:", ACOU_tr.shape)
print("Joint:", JOINT_tr.shape)
print("Quality:", QUALITY_tr.shape)

Define CV splitters and model factories (LogReg, Ridge, SVR, PCA+SVR, GBM, ExtraTrees) with grid searches

In [ ]:
outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
inner_s = StratifiedKFold(4, shuffle=True, random_state=SEED)
inner_k = KFold(4, shuffle=True, random_state=SEED)

clf_scorer = make_scorer(
    log_loss, labels=classes, response_method="predict_proba", greater_is_better=False
)

REG = "neg_root_mean_squared_error"

def words_clf():
    pipe = Pipeline([
        ("f", ColumnTransformer([
            ("tfidf", TfidfVectorizer(sublinear_tf=True, min_df=2), "text"),
            ("nw", StandardScaler(), ["n_words"])
        ])),
        ("clf", LogisticRegression(max_iter=3000))
    ])

    grid = {
        "f__tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
        "f__tfidf__max_features": [None, 5000, 10000],
        "clf__C": [0.03, 0.1, 0.3, 1, 3, 10, 30]
    }

    return GridSearchCV(pipe, grid, cv=inner_s, scoring=clf_scorer, n_jobs=-1)

def emb_clf(Cs):
    pipe = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=3000))])
    return lambda: GridSearchCV(pipe, {"clf__C": Cs}, cv=inner_s, scoring=clf_scorer, n_jobs=-1)

def ridge_reg():
    pipe = Pipeline([("sc", StandardScaler()), ("m", Ridge())])
    grid = {"m__alpha": np.logspace(-1, 6, 15)}      # extended upper range for bigger feature sets
    return GridSearchCV(pipe, grid, cv=inner_k, scoring=REG, n_jobs=-1)

def svr_reg():
    pipe = Pipeline([("sc", StandardScaler()), ("m", SVR(kernel="rbf"))])
    grid = {
        "m__C": [3, 10] if FAST else [0.3, 1, 3, 10, 30, 100],
        "m__epsilon": [0.1, 0.2] if FAST else [0.05, 0.1, 0.2, 0.3],
        "m__gamma": ["scale"] if FAST else ["scale", "auto"]
    }
    return GridSearchCV(pipe, grid, cv=inner_k, scoring=REG, n_jobs=-1)

# NEW: PCA before SVR, for high-dimensional inputs
def svr_pca_reg():
    pipe = Pipeline([
        ("sc", StandardScaler()),
        ("pca", PCA(random_state=SEED)),
        ("m", SVR(kernel="rbf"))
    ])
    grid = {
        "pca__n_components": [64] if FAST else [32, 64, 128],
        "m__C": [3, 10] if FAST else [1, 3, 10, 30],
        "m__epsilon": [0.1, 0.2],
        "m__gamma": ["scale"]
    }
    return GridSearchCV(pipe, grid, cv=inner_k, scoring=REG, n_jobs=-1)

def hgb_reg():
    dist = {
        "max_depth": [2, 3, 4, 5, None],
        "learning_rate": loguniform(0.01, 0.2),
        "max_iter": randint(100, 500),
        "min_samples_leaf": randint(5, 50),
        "l2_regularization": loguniform(1e-3, 20)
    }

    return RandomizedSearchCV(
        HistGradientBoostingRegressor(random_state=SEED),
        dist, n_iter=8 if FAST else 25, cv=inner_k,
        scoring=REG, n_jobs=-1, random_state=SEED
    )

# NEW: PCA before GBM, for high-dimensional inputs
def hgb_pca_reg():
    pipe = Pipeline([
        ("sc", StandardScaler()),
        ("pca", PCA(n_components=64, random_state=SEED)),
        ("m", HistGradientBoostingRegressor(random_state=SEED))
    ])
    dist = {
        "m__max_depth": [2, 3, 4, 5, None],
        "m__learning_rate": loguniform(0.01, 0.2),
        "m__max_iter": randint(100, 500),
        "m__min_samples_leaf": randint(5, 50),
        "m__l2_regularization": loguniform(1e-3, 20),
    }
    return RandomizedSearchCV(
        pipe, dist, n_iter=8 if FAST else 20, cv=inner_k,
        scoring=REG, n_jobs=-1, random_state=SEED
    )

def extra_reg():
    dist = {
        "n_estimators": randint(200, 700),
        "max_depth": [None, 8, 12, 16, 24],
        "min_samples_leaf": randint(1, 8),
        "max_features": ["sqrt", 0.5, 0.8, 1.0]
    }

    return RandomizedSearchCV(
        ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
        dist, n_iter=20, cv=inner_k, scoring=REG, n_jobs=-1, random_state=SEED
    )

def take(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]

def predict_score(search, kind, X):
    if kind == "clf":
        return to_score(search.predict_proba(X))
    return np.clip(search.predict(X), 0, 5)

Map each base model name to its type (clf/reg), factory, and feature matrices

In [ ]:
SPECS = {
    "words_clf":   ("clf", words_clf, Xw_tr, Xw_te),
    "text_clf":    ("clf", emb_clf([0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]), E_tr, E_te),
    "text_ridge":  ("reg", ridge_reg, E_tr, E_te),
    "text_svr":    ("reg", svr_pca_reg, E_tr, E_te),                 # CHANGED (PCA)

    "audio_clf":   ("clf", emb_clf([0.0001, 0.0003, 0.001, 0.003, 0.01]), A_w2v_tr, A_w2v_te),
    "audio_ridge": ("reg", ridge_reg, A_w2v_tr, A_w2v_te),
    "audio_svr":   ("reg", svr_pca_reg, A_w2v_tr, A_w2v_te),         # CHANGED (PCA)

    "whisper_ridge": ("reg", ridge_reg, WE_tr, WE_te),               # NEW
    "whisper_svr":   ("reg", svr_pca_reg, WE_tr, WE_te),             # NEW

    "quality_gbm": ("reg", hgb_reg, QUALITY_tr, QUALITY_te),

    "acoustic_ridge": ("reg", ridge_reg, ACOU_tr.values, ACOU_te.values),
    "acoustic_svr":   ("reg", svr_reg, ACOU_tr.values, ACOU_te.values),

    "joint_ridge": ("reg", ridge_reg, JOINT_tr, JOINT_te),
    "joint_svr":   ("reg", svr_pca_reg, JOINT_tr, JOINT_te),         # CHANGED (PCA)
    "joint_gbm":   ("reg", hgb_pca_reg, JOINT_tr, JOINT_te),         # CHANGED (PCA)
}

print("Number of base models:", len(SPECS))
print("\n".join(SPECS.keys()))

Run nested CV for every base model to get out-of-fold predictions (cached per model)

In [ ]:
# CHANGE: OOF results are cached per model. Finished models are skipped on re-run.
# If you change a model's INPUT features, delete its key first, e.g.:
#   for k in ["audio_clf", "audio_ridge"]: oof.pop(k, None)
OOF_CACHE = f"{WORK}/oof_v2.pkl"     # v2: wav2vec2 mean+std / PCA models are new, so old OOF is not reused
oof = pickle.load(open(OOF_CACHE, "rb")) if os.path.exists(OOF_CACHE) else {}

for name, (kind, mk, Xtr, _) in SPECS.items():
    if name in oof:
        print(f"skip {name} (cached, RMSE={rmse(y, oof[name]):.4f})")
        continue

    print("\n" + "=" * 70)
    print("MODEL:", name)

    o = np.zeros(len(y))

    for fold, (a, b) in enumerate(outer.split(np.zeros(len(y)), y_cls), 1):
        s = mk().fit(take(Xtr, a), y_cls[a] if kind == "clf" else y[a])
        o[b] = predict_score(s, kind, take(Xtr, b))
        print(f" fold {fold}: RMSE={rmse(y[b], o[b]):.4f}")

    oof[name] = o
    pickle.dump(oof, open(OOF_CACHE, "wb"))

    print(f"{name:18s} nested-CV RMSE={rmse(y, o):.4f} Pearson={pearsonr(y, o)[0]:.4f}")

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

Tabulate and plot RMSE/Pearson of all base models against the mean baseline

In [ ]:
results_df = pd.DataFrame([
    {"model": n, "RMSE": rmse(y, oof[n]), "Pearson": pearsonr(y, oof[n])[0]}
    for n in oof
]).sort_values("RMSE")

display(results_df)

plt.figure(figsize=(11, 6))
plt.barh(results_df["model"], results_df["RMSE"])
plt.axvline(base_rmse, linestyle="--", label="Mean baseline")
plt.xlabel("Nested-CV RMSE")
plt.title("Base model comparison")
plt.legend()
plt.tight_layout()
plt.show()

Compare NNLS/Ridge/Huber stackers by second-level CV, pick the best, and show its weights

In [ ]:
# CHANGE: compare blenders, pick the best one by honest second-level CV
names = list(SPECS.keys())
O = np.column_stack([oof[n] for n in names])
print("OOF matrix:", O.shape)

BLENDERS = {
    "nnls":  lambda: LinearRegression(positive=True),
    "ridge": lambda: Ridge(alpha=3.0, positive=True),
    "huber": lambda: HuberRegressor(alpha=1.0, max_iter=1000),
}

stack_cv = StratifiedKFold(5, shuffle=True, random_state=SEED + 1)

blend_oof, blend_rmse = {}, {}
for bname, mk_b in BLENDERS.items():
    p = np.zeros(len(y))
    for a, b in stack_cv.split(np.zeros(len(y)), y_cls):
        p[b] = mk_b().fit(O[a], y[a]).predict(O[b])
    p = np.clip(p, 0, 5)
    blend_oof[bname] = p
    blend_rmse[bname] = rmse(y, p)
    print(f"{bname:6s} stack CV RMSE={blend_rmse[bname]:.4f}")

best_blender_name = min(blend_rmse, key=blend_rmse.get)
print("Selected blender:", best_blender_name)

stack_oof = blend_oof[best_blender_name]
blender = BLENDERS[best_blender_name]().fit(O, y)

coef = getattr(blender, "coef_", None)
if coef is not None:
    print("Stack weights:")
    display(pd.Series(np.ravel(coef), index=names).sort_values(ascending=False))
print("Blender intercept:", round(float(np.ravel(blender.intercept_)[0]), 4))

Compare no/linear/isotonic calibration of the stacked predictions and report honest CV score

In [ ]:
# CHANGE: calibration (none / linear stretch / isotonic), chosen by CV
lin_oof = np.zeros(len(y))
iso_oof = np.zeros(len(y))

for a, b in KFold(5, shuffle=True, random_state=SEED + 2).split(stack_oof):
    lin_oof[b] = LinearRegression().fit(stack_oof[a, None], y[a]).predict(stack_oof[b, None])
    iso_oof[b] = IsotonicRegression(out_of_bounds="clip").fit(stack_oof[a], y[a]).predict(stack_oof[b])

cal_rmse = {
    "none": rmse(y, stack_oof),
    "linear": rmse(y, np.clip(lin_oof, 0, 5)),
    "isotonic": rmse(y, iso_oof),
}
print({k: round(v, 4) for k, v in cal_rmse.items()})

# require a clear gain (0.003) before switching away from "none"
best_cal = "none"
for k in ["linear", "isotonic"]:
    if cal_rmse[k] < cal_rmse[best_cal] - 0.003:
        best_cal = k
print("Selected calibration:", best_cal)

final_cv_oof = {"none": stack_oof, "linear": np.clip(lin_oof, 0, 5), "isotonic": iso_oof}[best_cal]

cv_rmse = rmse(y, final_cv_oof)
cv_r = pearsonr(y, final_cv_oof)[0]

print("=" * 60)
print("HONEST STACK PERFORMANCE")
print("Nested-CV RMSE  :", round(cv_rmse, 4))
print("Nested-CV Pearson:", round(cv_r, 4))
print("Baseline RMSE   :", round(base_rmse, 4))
print("=" * 60)

Plot predicted vs true, residual histogram, and per-model RMSE vs the stack

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

ax[0].scatter(y, final_cv_oof, alpha=0.45)
ax[0].plot([0, 5], [0, 5], linestyle="--")
ax[0].set(xlabel="True label", ylabel="Predicted", title="Nested-CV prediction vs true")

sns.histplot(y - final_cv_oof, bins=30, kde=True, ax=ax[1])
ax[1].set_title("Stack residuals")

all_names = names + ["STACK"]
all_rmse = [rmse(y, oof[n]) for n in names] + [cv_rmse]

ax[2].bar(all_names, all_rmse)
ax[2].axhline(base_rmse, linestyle="--", label="Mean baseline")
ax[2].set(ylabel="Nested-CV RMSE", title="Model comparison")
ax[2].tick_params(axis="x", rotation=65)
ax[2].legend()

plt.tight_layout()
plt.show()

print("Mean residual:", round(float(np.mean(y - final_cv_oof)), 4))
print("Residual std :", round(float(np.std(y - final_cv_oof)), 4))

Refit all base models on full train, blend and calibrate to produce test predictions

In [ ]:
# Final fit of all base models (cached to disk so a crash does not lose it)
FINAL_CACHE = f"{WORK}/final_scores_v2.pkl"
final_scores = pickle.load(open(FINAL_CACHE, "rb")) if os.path.exists(FINAL_CACHE) else {}

for name, (kind, mk, Xtr, Xte) in SPECS.items():
    if name in final_scores:
        print("skip final:", name)
        continue

    print("\nFitting final:", name)
    s = mk().fit(Xtr, y_cls if kind == "clf" else y)
    print("best params:", s.best_params_)

    final_scores[name] = (predict_score(s, kind, Xtr), predict_score(s, kind, Xte))
    pickle.dump(final_scores, open(FINAL_CACHE, "wb"))

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

tr_scores = np.column_stack([final_scores[n][0] for n in names])
te_scores = np.column_stack([final_scores[n][1] for n in names])

train_pred = np.clip(blender.predict(tr_scores), 0, 5)
test_pred = np.clip(blender.predict(te_scores), 0, 5)

# apply the selected calibration (fitted on honest OOF stack predictions)
if best_cal == "linear":
    calibrator = LinearRegression().fit(stack_oof[:, None], y)
    test_pred = np.clip(calibrator.predict(test_pred[:, None]), 0, 5)
elif best_cal == "isotonic":
    calibrator = IsotonicRegression(out_of_bounds="clip").fit(stack_oof, y)
    test_pred = np.clip(calibrator.predict(test_pred), 0, 5)

print("=" * 65)
print(f"TRAINING RMSE (not honest): {rmse(y, train_pred):.4f}")
print(f"NESTED-CV RMSE : {cv_rmse:.4f}")
print(f"NESTED-CV Pearson: {cv_r:.4f}")
print(f"BASELINE RMSE  : {base_rmse:.4f}")
print("=" * 65)

print("\nTest prediction distribution:")
print(pd.Series(test_pred).describe())

Write submission.csv and sanity-check its shape, uniqueness and NaNs (assert failed here, see note)

In [ ]:
test_pred = np.asarray(test_pred).reshape(-1)

assert len(test_pred) == len(test), f"Predictions: {len(test_pred)}, test rows: {len(test)}"

sub = pd.DataFrame({
    "filename": test["filename"].values,
    "label": np.clip(test_pred, 0, 5)
})

submission_path = f"{WORK}/submission.csv"
sub.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Submission shape:", sub.shape)
print("Columns:", sub.columns.tolist())
print("Unique filenames:", sub["filename"].nunique())

display(sub.head(10))
display(sub["label"].describe())

assert len(sub) == len(sample_sub)
assert sub["filename"].is_unique
assert sub["label"].notna().all()

Plot histogram of the final test predictions

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(test_pred, bins=20)
plt.xlabel("Predicted score")
plt.ylabel("Count")
plt.title("Test prediction distribution")
plt.show()